# Session 16: Build a Ticket Agent with a Saved Classifier

Reuse one classifier from Sessions 14–15. The saved classifier assigns a category. Qwen decides whether the message needs classification and writes the reply.

## 1. Choose a saved classifier

First save a classifier in the [Sessions 14–15 practice notebook](https://colab.research.google.com/github/wlos/AI_SS/blob/main/14-15-Sequence-Models-and-Transformers/practice/Session14-15%20Support%20Ticket%20Classification%20Practice.ipynb).

In Colab, select a **T4 GPU** for Qwen and upload `saved-ticket-classifier.zip`, `saved-gpt-classifier.zip` or `saved-count-mlp.zip` from the earlier practice. Set `CLASSIFIER_PATH` to that ZIP or a local saved model folder. For the MLP, use a joblib file you saved yourself in an environment with the same scikit-learn version.

In [ ]:
import os
import subprocess
import sys

import torch

CLASSIFIER_PATH = "saved-ticket-classifier.zip"
if not os.path.exists(CLASSIFIER_PATH):
    raise FileNotFoundError(f"Upload {CLASSIFIER_PATH} or set CLASSIFIER_PATH to the saved folder.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1"], check=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Qwen device: {device}")

## 2. Load the classifier

The saved DistilBERT and GPT-2 folders include the model, its label mapping and the tokenizer. The MLP bundle includes its word-count vectoriser, the fitted classifier and the label names. All three variants return one category to the agent. The classifier runs on CPU so Qwen can use the GPU.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

with TemporaryDirectory() as unpacked:
    source = CLASSIFIER_PATH
    if CLASSIFIER_PATH.endswith(".zip"):
        # Extract only files needed to load the saved classifier.
        model_files = {
            "classifier.joblib", "config.json", "model.safetensors",
            "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
            "vocab.json", "vocab.txt", "merges.txt", "added_tokens.json",
        }
        with ZipFile(CLASSIFIER_PATH) as archive:
            archive.extractall(unpacked, members=model_files.intersection(archive.namelist()))
        source = unpacked

    mlp_path = f"{source}/classifier.joblib"
    saved_mlp = None
    if os.path.isfile(mlp_path):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib"], check=True)
        import joblib

        saved_mlp = joblib.load(mlp_path)
    else:
        # Load both the classification head and its matching tokenizer from disk.
        classifier = pipeline(
            "text-classification",
            model=AutoModelForSequenceClassification.from_pretrained(
                source, local_files_only=True, use_safetensors=True,
            ),
            tokenizer=AutoTokenizer.from_pretrained(source, local_files_only=True),
            device=-1,
        )


def classify_ticket(ticket):
    if saved_mlp is not None:
        # Reuse the fitted vocabulary so feature columns match the training data.
        counts = saved_mlp["vectorizer"].transform([ticket])
        prediction = saved_mlp["classifier"].predict(counts)[0]
        category = saved_mlp["class_names"][int(prediction)]
    else:
        category = classifier(ticket, truncation=True, max_length=96)[0]["label"]
    return category if category in CATEGORIES else None

## 3. Load the support guides

The agent loads the [support guides](https://wlos.github.io/AI_SS/16-LLM-Fine-Tuning-and-Agents/sample_data/support_guides.json) from a local JSON file. Colab downloads it on the first run. To use the bundled guides from the local practice folder, set `GUIDE_JSON` to `../sample_data/support_guides.json`.

In [ ]:
import json
from urllib.request import urlretrieve

GUIDE_JSON = "support_guides.json"
# Download once so edits to the local guides are kept when this cell is rerun.
if not os.path.isfile(GUIDE_JSON):
    urlretrieve(
        "https://raw.githubusercontent.com/wlos/AI_SS/main/16-LLM-Fine-Tuning-and-Agents/sample_data/support_guides.json",
        GUIDE_JSON,
    )

with open(GUIDE_JSON, encoding="utf-8") as guide_file:
    support_guides = json.load(guide_file)

CATEGORIES = ("Account access", "Hardware", "Network", "Software")

## 4. Load Qwen for routing and replies

Load the original Qwen instruct checkpoint. The separate classifier handles the category prediction.

In [ ]:
from transformers import AutoModelForCausalLM, GenerationConfig

CHECKPOINT = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
dtype = torch.float16 if device.type == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(CHECKPOINT, dtype=dtype).to(device)
model.eval()


generation_settings = GenerationConfig(
    do_sample=False,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)


def encode_chat(messages):
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)


@torch.inference_mode()
def ticket_needs_classifier(ticket):
    messages = [{
        "role": "system",
        "content": (
            "You route messages to an IT service desk. Reply Yes for a specific technical problem "
            "or a request for troubleshooting, and No for small talk or a general knowledge question. "
            "Give only one label, Yes or No."
        ),
    }]
    examples = [
        ("Good morning. What can you do?", "No"),
        ("I am locked out of the course portal.", "Yes"),
        ("What is a computer network?", "No"),
        ("How do I reconnect to the campus network?", "Yes"),
        ("Thanks for your help!", "No"),
        ("My laptop will not charge.", "Yes"),
    ]
    for message, answer in examples:
        messages.extend([
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer},
        ])
    messages.append({"role": "user", "content": ticket})

    # Choose between the two allowed next tokens without generating extra text.
    options = tokenizer.convert_tokens_to_ids(["Yes", "No"])
    inputs = encode_chat(messages)
    logits = model(**inputs, use_cache=False, logits_to_keep=1).logits
    return logits[0, -1, options].argmax().item() == 0


@torch.inference_mode()
def generate_reply(prompt, max_new_tokens=64):
    inputs = encode_chat([{"role": "user", "content": prompt}])
    tokens = model.generate(**inputs, generation_config=generation_settings, max_new_tokens=max_new_tokens)
    # Keep only the generated reply, leaving the input prompt out of the chat.
    reply = tokens[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 5. Run the agent

Qwen compares its next-token scores for **Yes** and **No**. Yes calls the classifier and No gives a direct reply. If the classifier returns an allowed category, Python looks up the matching guide and returns its solution. If classification fails, ask once for more details and retry with the updated ticket.

In [ ]:
def run_ticket_agent(ticket, guides, *, is_follow_up=False):
    trace = {
        "decision": "CLASSIFY",
        "category": None,
        "guide": None,
        "needs_clarification": False,
    }

    # A reply to a clarification stays on the classification path.
    if not is_follow_up and not ticket_needs_classifier(ticket):
        trace["decision"] = "DIRECT"
        trace["reply"] = generate_reply(
            "Reply briefly as a help desk assistant. "
            "You can help with account access, hardware, network and software issues.\n"
            f"Message: {ticket}"
        )
        return trace

    category = classify_ticket(ticket)
    trace["category"] = category
    if category is None:
        if is_follow_up:
            trace["reply"] = "I could not classify this ticket. Please contact the service desk."
        else:
            trace["decision"] = "ASK"
            trace["needs_clarification"] = True
            trace["reply"] = "Which service or device is affected, and what happened?"
        return trace

    # Look up the solution using the category as the JSON key.
    guide = guides.get(category)
    trace["guide"] = guide
    trace["reply"] = guide or "I could not find a guide for this issue. Please contact the service desk."
    return trace

## 6. Chat with the ticket agent

Run the chat cell and enter these messages one at a time. Compare the displayed decision and category with the expected results. For a classified ticket, the chat displays the matching guide as the solution.

| Message | Decision | Category |
| --- | --- | --- |
| Hello, what can you help with? | DIRECT | — |
| My student portal account is locked. | CLASSIFY | Account access |
| My laptop shuts down when its battery is unplugged from the charger. | CLASSIFY | Hardware |

If the classifier returns no valid category, the chat shows `ASK`. Reply with the affected service or device and what happened. The agent retries classification with both messages. After a final reply, the next message starts a new ticket.

Type `/exit` to stop. Inspect `last_trace` to see the decision, category and guide.

In [ ]:
ticket_context = []
last_trace = None
print("Ticket support chat. Type /exit to stop.")

while True:
    message = input("You: ").strip()
    if message.casefold() == "/exit":
        break
    if not message:
        continue

    ticket_context.append(message)
    last_trace = run_ticket_agent(
        " ".join(ticket_context), support_guides,
        is_follow_up=len(ticket_context) > 1,
    )
    # Show the agent decision and any category returned by the classifier.
    print(f"Decision: {last_trace['decision']}")
    if last_trace["category"] is not None:
        print(f"Category: {last_trace['category']}")
    reply_label = "Solution" if last_trace["guide"] else "Agent"
    print(f"{reply_label}: {last_trace['reply']}")

    # Keep earlier details while the agent is waiting for clarification.
    if not last_trace["needs_clarification"]:
        ticket_context.clear()